# Countries, Entities, and groups
This Jupyter notebook will elaborate on the creation of the dataset

## Chapter 1: Our World in Data

First we want to create a dataset of all countries, their ISO codes and regions from Our World in Data. Our main sources would be the OWID's [World map region definitions](https://ourworldindata.org/world-region-map-definitions) page and OWID's [standard entity names](https://github.com/owid/energy-data/tree/master/scripts/input/shared). ([source](https://raw.githubusercontent.com/owid/energy-data/master/scripts/input/shared/iso_codes.csv)). 

However, these sources did not list all available OWID's entities, so we will also interpolate our dataset with entities listed in OWID's [Energy Data](https://github.com/owid/energy-data) ([source](https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv)), and [CO2 Data](https://github.com/owid/co2-data) ([source](https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv))

### World map region definitions

First, we manually download the datasets from https://ourworldindata.org/world-region-map-definitions and stored them in the "owid" directory

Then, we read those downloaded csv file, store each csv table in a DataFrame, and merge the DataFrames together using the Code attribute

In [1]:
# import the needed packages
import pandas as pd
import os

In [ ]:
# read the files

directory = "owid"
datafiles = []

for dirpath, _, filenames in os.walk(directory):
    for f in filenames:
        datafiles.append(os.path.join(dirpath, f))

datafiles

In [ ]:
# load each file into a DataFrame

dfs = []

for f in datafiles:
    # drop the Year column since we don't need it
    df = pd.read_csv(f).drop('Year', axis=1)
    dfs.append(df)

dfs

In [ ]:
# merge all dataframes based on Entity and Code

df_merged = dfs[0]

for df in dfs[1:]:
    df_merged = pd.merge(df_merged, df, on=["Entity", "Code"], how="outer")

df_merged

However, not all OWID countries are listed in https://ourworldindata.org/world-region-map-definitions. We need to 

In [6]:
df = pd.read_csv("https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv")

df = df[["iso_code", "country"]].drop_duplicates()

df_owid_energy_data = df.set_index(["iso_code", "country"])

df_owid_energy_data


,
iso_code,country
AFG,Afghanistan
OWID_AFR,Africa
ALB,Albania
DZA,Algeria
ASM,American Samoa
...,...
OWID_WRL,World
YEM,Yemen
NaN,Yugoslavia


In [ ]:

# read the OWID CO2 data
df = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")

df = df[["iso_code", "country"]].drop_duplicates()

df_owid_co2_data = df.set_index(["iso_code", "country"])

df_owid_data = pd.merge(df_owid_energy_data, df_owid_co2_data,
                        how="outer", left_index=True, right_index=True)

df = pd.read_csv(
    "https://raw.githubusercontent.com/owid/energy-data/master/scripts/input/shared/iso_codes.csv")

df = df.drop_duplicates(subset="iso_code")

df.rename(columns={'Country': 'country'}, inplace=True)

df_owid_country = df.set_index(["iso_code", "country"])

df_owid = pd.merge(df_owid_data, df_owid_country, how="outer",
                    left_index=True, right_index=True)

df_owid

In [7]:
df_merged

,Entity,Code,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations
0,Abkhazia,OWID_ABK,Asia,NaN,NaN,NaN
1,Afghanistan,AFG,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia
2,Akrotiri and Dhekelia,OWID_AKD,Asia,NaN,NaN,NaN
3,Albania,ALB,Europe,Europe,Europe and Central Asia,Europe and Northern America
4,Algeria,DZA,Africa,Africa,Middle East and North Africa,Northern Africa and Western Asia
...,...,...,...,...,...,...
283,Zimbabwe,ZWE,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa
284,Åland Islands,ALA,Europe,NaN,NaN,NaN
285,Micronesia,NaN,NaN,NaN,East Asia and Pacific,Oceania
286,Aland Islands,NaN,NaN,NaN,NaN,Europe and Northern America


In [ ]:
import world_bank_data as wb
import weo